# Figure 4 — Physical stability

Global pressure, moisture, and energy behavior.


## 1. Load only the required common-grid data


In [ ]:
from pathlib import Path
import importlib
import sys
import numpy as np
import xarray as xr
import dask
from dask.distributed import Client, get_client

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run from ml_implement_paper/ or its notebooks/ directory')
sys.path.insert(0, str(PROJECT_ROOT))

from src import io as data_io
from src import metrics, plotting, preprocessing
plotting = importlib.reload(plotting)

config = data_io.load_config(PROJECT_ROOT / 'config' / 'paths.yaml')
analysis = config['analysis']

# ------------------------- User-adjustable settings -------------------------
OUTPUT_ROOT = Path('/global/cfs/cdirs/e3sm/www/zhan391/sea_crogs/online_diag')
ML_ROOT = Path('/pscratch/sd/z/zhan391/seacrogs_scratch/ml_method_2026')
REFERENCE_ROOT = Path('/pscratch/sd/z/zhan391/seacrogs_scratch/reference_nudge')
POST_SUBDIR = Path('post/atm/180x360_aave/ts/3hourly/1yr')
PERIOD = '201201_201212'
FORCE_COMPUTE = False  # True: overwrite selected case caches.
REQUIRED_VARIABLES = ['PS', 'TMQ', 'FSNT', 'FLNT']
paths = {
    'processed': OUTPUT_ROOT / 'processed',
    'figures': OUTPUT_ROOT / 'figures',
    'tables': OUTPUT_ROOT / 'tables',
}
for output_dir in paths.values():
    output_dir.mkdir(parents=True, exist_ok=True)
# Select cases here: keep CTRL and comment out any ML case you do not want.
CASE_DIRS = {
    'CTRL': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_CTRL',
    'UNET-IMT': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNET_IMT_NS6_WOQMADJ_WOTVCON_PBL222_WOVSMOOTH',
    'UNETXTR-IMT': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_WOQMADJ_WOTVCON_PBL222_WOVSMOOTH',
    'UNETXTR-IMT-A15': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'UNETXTR-IMT-C05': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_SCL0.5_WOQMADJ_WOTVCON_PBL222_WOVSMOOTH',
    'UNETXTR-IMT-C05A15': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_SCL0.5_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'UNETXTR-LCZ-C05A15': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_LANCZOS_NS6_SCL0.5_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'UNETXTR-LCZ-C05A15-PTAP100': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_LANCZOS_NS6_SCL0.5_PTAPUNI100HPA_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'REF': REFERENCE_ROOT / 'F20TR_ne30pg2_EC30to60E2r2_NDGUVTQ_IMT_3hr_pm-cpu_08-01-25',
}
FONT_SIZE = 14
FIGURE_SIZE = (11, 7)
LINE_WIDTH = 1.5
LEGEND_FRAME = False
FIGURE_TITLE = 'Physical stability'
STATE_FIELDS = [
    ('PS_anomaly', 'Surface pressure anomaly'),
    ('TMQ_anomaly', 'TMQ anomaly'),
    ('TOA_imbalance', 'FSNT − FLNT'),
]
COLORS = {
    'CTRL': '#222222', 'UNET-IMT': '#2878B5', 'UNETXTR-IMT': '#D95319',
    'UNETXTR-IMT-A15': '#9467BD', 'UNETXTR-IMT-C05': '#2CA02C',
    'UNETXTR-IMT-C05A15': '#8C564B', 'UNETXTR-LCZ-C05A15': '#17BECF',
    'UNETXTR-LCZ-C05A15-PTAP100': '#E377C2',
    'REF': '#7F7F7F',
}
# ---------------------------------------------------------------------------

if 'CTRL' not in CASE_DIRS:
    raise ValueError('CASE_DIRS must include CTRL')
ANALYSIS_CASES = tuple(CASE_DIRS)
DIAGNOSTIC_DIR = paths['processed'] / 'stability_timeseries'
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTIC_PATHS = {name: DIAGNOSTIC_DIR / f'{name}_{PERIOD}.nc' for name in ANALYSIS_CASES}
EXPECTED_FIELDS = {'PS_anomaly', 'TMQ_anomaly', 'TOA_imbalance'}

def cache_is_compatible(path, experiment):
    if not path.exists():
        return False
    try:
        with xr.open_dataset(path) as cached:
            experiments = set(cached['experiment'].values.astype(str))
            return experiment in experiments and EXPECTED_FIELDS.issubset(cached.data_vars)
    except (KeyError, OSError, ValueError):
        return False

CASES_TO_COMPUTE = tuple(
    name for name, path in DIAGNOSTIC_PATHS.items()
    if FORCE_COMPUTE or not cache_is_compatible(path, name)
)
RECOMPUTE = bool(CASES_TO_COMPUTE)

# Reuse an existing client or start one if the diagnostic must be computed.
client = None
if RECOMPUTE:
    try:
        client = get_client()
    except ValueError:
        client = Client(n_workers=4, threads_per_worker=1, processes=False, dashboard_address=':0')
STATE_VARIABLES = ['PS', 'TMQ', 'FSNT', 'FLNT']
REQUIRED_BY_CASE = {name: STATE_VARIABLES for name in CASE_DIRS}
files = {
    name: {
        variable: sorted((case_dir / POST_SUBDIR).glob(f'{variable}_*{PERIOD}.nc'))
        for variable in REQUIRED_BY_CASE[name]
    }
    for name, case_dir in CASE_DIRS.items() if name in CASES_TO_COMPUTE
} if RECOMPUTE else {}
missing = {
    name: [variable for variable, matches in variable_files.items() if not matches]
    for name, variable_files in files.items()
}
missing = {name: variables for name, variables in missing.items() if variables}
if missing:
    details = '\n'.join(f"  {name}: {', '.join(variables)}" for name, variables in missing.items())
    raise FileNotFoundError('Postprocess the required variables first:\n' + details)

chunks = {'time': 32, 'lat': 45, 'lon': 90}
datasets = {
    name: xr.merge(
        [
            xr.open_dataset(path, chunks=chunks, cache=False)
            for matches in variable_files.values() for path in matches
        ],
        join='exact', compat='no_conflicts',
    )
    for name, variable_files in files.items()
} if RECOMPUTE else None
datasets = {
    name: preprocessing.subset_time(ds, analysis['start_date'], analysis['end_date'])
    for name, ds in datasets.items()
} if RECOMPUTE else None
datasets = preprocessing.match_common_times(datasets) if RECOMPUTE else None
datasets = {name: preprocessing.daily_mean(ds) for name, ds in datasets.items()} if RECOMPUTE else None

if RECOMPUTE:
    sample = datasets[next(iter(datasets))][REQUIRED_VARIABLES[0]].isel(time=0, drop=True)
    area = np.cos(np.deg2rad(sample['lat'])).clip(min=0).broadcast_like(sample)
    area = area / area.sum()
    status = {'mode': 'compute', 'cases': CASES_TO_COMPUTE, 'datasets': {name: dict(ds.sizes) for name, ds in datasets.items()}}
else:
    status = {'mode': 'cached', 'paths': {name: str(path) for name, path in DIAGNOSTIC_PATHS.items()}}
status

## 2. Quality control


In [ ]:
# Run all finite-value checks together so Dask can share I/O efficiently.

qc_keys = []
qc_tasks = []
for case_name, dataset in datasets.items() if RECOMPUTE else []:
    for variable in REQUIRED_BY_CASE[case_name]:
        qc_keys.append((case_name, variable))
        qc_tasks.extend([
            np.isfinite(dataset[variable]).any().data,
            (~np.isfinite(dataset[variable])).sum().data,
        ])
qc_values = dask.compute(*qc_tasks)
qc = {}
for index, key in enumerate(qc_keys):
    has_finite = bool(qc_values[2 * index])
    invalid_count = int(qc_values[2 * index + 1])
    if not has_finite:
        raise ValueError(f'{key[0]}:{key[1]} contains no finite values')
    qc.setdefault(key[0], {})[key[1]] = invalid_count
qc

## 3. Process and save the diagnostic data


In [ ]:
if RECOMPUTE:
    for name in CASES_TO_COMPUTE:
        case_diagnostic = metrics.stability_diagnostic({name: datasets[name]}, area, tendencies=[])
        invalid_outputs = [
            field for field, values in case_diagnostic.data_vars.items()
            if not bool(np.isfinite(values).any().compute())
        ]
        if invalid_outputs:
            raise ValueError(f'{name} stability fields contain no finite data: ' + ', '.join(invalid_outputs))
        case_diagnostic.attrs.update({
            'experiment': name, 'period': PERIOD, 'case_directory': str(CASE_DIRS[name]),
        })
        data_io.save_dataset(case_diagnostic, DIAGNOSTIC_PATHS[name])

diagnostic = xr.concat(
    [xr.open_dataset(DIAGNOSTIC_PATHS[name]) for name in ANALYSIS_CASES], dim='experiment'
)
diagnostic

## 4. Reload the diagnostic product and create the figure


In [ ]:
FIGURE_PATH = paths['figures'] / 'fig04_stability.png'
PLOT_OPTIONS = {
    'figsize': FIGURE_SIZE,
    'colors': COLORS,
    'title': FIGURE_TITLE,
    'state_fields': STATE_FIELDS,
    'legend_frame': LEGEND_FRAME, 'linewidth': LINE_WIDTH,
    'font_size': FONT_SIZE,
}
diagnostic = xr.concat(
    [xr.open_dataset(DIAGNOSTIC_PATHS[name]) for name in ANALYSIS_CASES], dim='experiment'
)
plotting.plot_stability(diagnostic, FIGURE_PATH, **PLOT_OPTIONS)